In [0]:
CREATE OR REPLACE TABLE topcars_sa.gold.dim_dealer AS
SELECT 
   ROW_NUMBER() OVER(ORDER BY DealerID) AS DealerKey,
   DealerID AS DealerID_natural,
   DealerName,
   City,
   Country 
FROM topcars_sa.silver.dealer_silver;



CREATE OR REPLACE TABLE topcars_sa.gold.dim_model AS
SELECT 
   ROW_NUMBER() OVER(ORDER BY ModelID) AS ModelKey,
   ModelID AS ModelID_natural,
   Brand,
   Model,
   Segment,
   EngineSize_L,
   Fuel
FROM topcars_sa.silver.models_silver;


CREATE OR REPLACE TABLE topcars_sa.gold.dim_date AS
SELECT
  ROW_NUMBER() OVER (ORDER BY calendar_date) AS DateKey,
  calendar_date AS FullDate,
  YEAR(calendar_date) AS Year,
  MONTH(calendar_date) AS Month,
  DAY(calendar_date) AS Day,
  DATE_FORMAT(calendar_date, 'MMMM') AS MonthName,
  DATE_FORMAT(calendar_date, 'EEEE') AS DayName,
  QUARTER(calendar_date) AS Quarter
FROM (
  SELECT explode(sequence(to_date('2022-01-01'), to_date('2025-12-31'), interval 1 day)) AS calendar_date
);

CREATE OR REPLACE TABLE topcars_sa.gold.fact_sales AS
SELECT 
   s.SaleID,
   d.DealerKey,
   m.ModelKey,
   dt.DateKey,
   s.Quantity,
   s.TotalPrice_ZAR,
   s.Total_Profit_ZAR
FROM topcars_sa.silver.sales_silver s
JOIN topcars_sa.gold.dim_dealer d ON s.DealerID = d.DealerID_natural
JOIN topcars_sa.gold.dim_model m ON s.ModelID = m.ModelID_natural
JOIN topcars_sa.gold.dim_date dt ON s.Date = dt.FullDate;
